In [0]:
%run "../../utils/00_config"

In [0]:
%run "../../utils/02_data_quality"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

raw_responses = spark.table(
    get_table("bronze", "raw_api_responses")
)

orders_raw = (
    raw_responses
    .filter(F.col("event_type") == "orders")
    .select(
        "execution_id",
        "observed_at",
        "product_name",
        "product_slug",
        "category",
        "resource_slug",
        F.col("response.data").alias("orders_data")
    )
)

display(orders_raw)

In [0]:
user_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("ingameName", T.StringType()),
    T.StructField("slug", T.StringType()),
    T.StructField("reputation", T.IntegerType()),
    T.StructField("platform", T.StringType()),
    T.StructField("crossplay", T.BooleanType()),
    T.StructField("locale", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("activity", T.StructType([
        T.StructField("type", T.StringType()),
        T.StructField("details", T.StringType())
    ])),
    T.StructField("lastSeen", T.StringType())
])

order_schema = T.ArrayType(
    T.StructType([
        T.StructField("id", T.StringType()),
        T.StructField("type", T.StringType()),
        T.StructField("platinum", T.IntegerType()),
        T.StructField("quantity", T.IntegerType()),
        T.StructField("perTrade", T.IntegerType()),
        T.StructField("visible", T.BooleanType()),
        T.StructField("createdAt", T.StringType()),
        T.StructField("updatedAt", T.StringType()),
        T.StructField("itemId", T.StringType()),
        T.StructField("user", user_schema)
    ])
)

In [0]:
orders_parsed = (
    orders_raw
    .withColumn(
        "orders",
        F.from_json("orders_data", order_schema)
    )
)

orders_exploded = (
    orders_parsed
    .withColumn("order", F.explode("orders"))
)

In [0]:
stg_wfm_orders = orders_exploded.select(
    "execution_id",
    F.to_timestamp("observed_at").alias("observed_at"),
    "product_name",
    "product_slug",
    "category",
    F.col("resource_slug").alias("item_slug"),

    F.col("order.id").alias("order_id"),
    F.col("order.itemId").alias("item_id"),
    F.col("order.type").alias("order_type"),
    F.col("order.platinum").alias("platinum"),
    F.col("order.quantity").alias("quantity"),
    F.col("order.perTrade").alias("per_trade"),
    F.col("order.visible").alias("is_visible"),

    F.to_timestamp("order.createdAt").alias("order_created_at"),
    F.to_timestamp("order.updatedAt").alias("order_updated_at"),

    F.col("order.user.id").alias("player_id"),
    F.col("order.user.ingameName").alias("player_name"),
    F.col("order.user.slug").alias("player_slug"),
    F.col("order.user.reputation").alias("player_reputation"),
    F.col("order.user.platform").alias("platform"),
    F.col("order.user.crossplay").alias("crossplay"),
    F.col("order.user.locale").alias("locale"),
    F.col("order.user.status").alias("player_status"),
    F.col("order.user.activity.type").alias("player_activity_type"),
    F.col("order.user.activity.details").alias("player_activity_details"),
    F.to_timestamp("order.user.lastSeen").alias("player_last_seen")
)

display(stg_wfm_orders)

In [0]:
# Data quality checks

check_row_count(stg_wfm_orders, 3937)

check_required_fields(
    stg_wfm_orders,
    [
        "execution_id",
        "order_id",
        "item_id",
        "player_id",
        "item_slug",
        "order_type"
    ]
)

check_unique_keys(
    stg_wfm_orders,
    ["execution_id", "order_id"]
)

# Validate order types
order_types = {
    row["order_type"]
    for row in stg_wfm_orders.select("order_type").distinct().collect()
}

assert order_types == {"buy", "sell"}, (
    f"Unexpected order types: {order_types}"
)

print("All staging validations passed.")

In [0]:
display(
    stg_wfm_orders
    .groupBy("order_type")
    .count()
    .orderBy("order_type")
)

In [0]:
# Materialize staging table

from delta.tables import DeltaTable

target_table = get_table("staging", "stg_wfm_orders")

merge_condition = """
    target.execution_id = source.execution_id
    AND target.order_id = source.order_id
"""

if spark.catalog.tableExists(target_table):
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(
            stg_wfm_orders.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        stg_wfm_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

print(f"Table materialized: {target_table}")

In [0]:
# Validate materialized table

stg_orders_saved = spark.table(target_table)

print(f"Total records: {stg_orders_saved.count()}")
print(
    f"Distinct orders: "
    f"{stg_orders_saved.select('execution_id', 'order_id').distinct().count()}"
)

display(
    stg_orders_saved
    .groupBy("order_type")
    .count()
    .orderBy("order_type")
)